# 01 — 3D HAMSTER Planner 环境与模型加载

目标：在 Colab 中创建独立的 /content/.venv_planner，安装固定 commit 的官方推理代码，下载固定 revision 的 checkpoint，并完成环境、CUDA 与模型加载检查。

Notebook 默认内核不会切换到 Planner 环境。除安装 uv 外，后续所有 Python 命令都显式调用 /content/.venv_planner/bin/python。本文件未保存任何执行输出；请在 A100（优先 80GB）运行时执行。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/vla-3d-trajectory-guidance")).resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "planner_colab.json"
assert CONFIG_PATH.is_file(), (
    f"未找到 {CONFIG_PATH}。请先把完整项目放到 /content/vla-3d-trajectory-guidance，"
    "或设置 PROJECT_ROOT 环境变量。"
)

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
PLANNER_PY = Path(config["planner"]["env_dir"]) / "bin" / "python"
CHECKOUT_DIR = Path(config["planner"]["checkout_dir"])
CHECKPOINT_DIR = Path(config["planner"]["checkpoint_dir"])

print("Project:", PROJECT_ROOT)
print("Planner interpreter:", PLANNER_PY)
print("3D HAMSTER checkout:", CHECKOUT_DIR)
print("Checkpoint:", CHECKPOINT_DIR)
print("Pinned code commit:", config["planner"]["repo_commit"])
print("Pinned model revision:", config["planner"]["model_revision"])


## 1. 检查 Colab GPU、驱动、显存和磁盘

nvidia-smi 显示的是驱动支持的 CUDA 上限，不等于 PyTorch 的构建 CUDA 版本。真正的 PyTorch CUDA 版本会在目标环境中再次检查。

In [ ]:
subprocess.run(["nvidia-smi"], check=True)
gpu_csv = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,driver_version,memory.total,memory.free",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
print("GPU summary:", gpu_csv)
disk = shutil.disk_usage("/content")
print(f"/content free: {disk.free / 1024**3:.1f} GiB")

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
ffmpeg_version = subprocess.check_output(["ffmpeg", "-version"], text=True).splitlines()[0]
print(ffmpeg_version)


## 2. 只在默认 Notebook 内核安装固定版本 uv

这里不向默认内核安装 Torch 或模型依赖。uv 只负责创建和管理目标虚拟环境。

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", f"uv=={config['uv_version']}"],
    check=True,
)
uv_bin = shutil.which("uv")
assert uv_bin, "uv 安装后仍不在 PATH，请重启运行时并重新执行本 Notebook。"
uv_version = subprocess.check_output([uv_bin, "--version"], text=True).strip()
assert config["uv_version"] in uv_version, (uv_version, config["uv_version"])
print(uv_version)


## 3. 创建独立 Planner 环境并安装固定上游代码

脚本会拒绝复用 Python 版本不符的旧环境，保护参考核心版本，并在上游 checkout 有本地改动时停止。默认不安装可选 xformers，避免它重新解析 Torch。

In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "bootstrap_planner_env.py"),
        "--project-root",
        str(PROJECT_ROOT),
        "--config",
        str(CONFIG_PATH),
    ],
    check=True,
)
assert PLANNER_PY.is_file(), PLANNER_PY


## 4. 用目标解释器做精确版本、导入与 CUDA 冒烟检查

In [ ]:
def planner_subprocess_env():
    env = os.environ.copy()
    npp_dir = Path(config["planner"]["env_dir"]) / "lib" / "python3.12" / "site-packages" / "nvidia" / "npp" / "lib"
    current = env.get("LD_LIBRARY_PATH", "")
    env["LD_LIBRARY_PATH"] = str(npp_dir) + (":" + current if current else "")
    return env

def run_planner_script(script_name, *arguments, check=True):
    command = [
        str(PLANNER_PY),
        str(PROJECT_ROOT / "scripts" / script_name),
        "--project-root",
        str(PROJECT_ROOT),
        "--config",
        str(CONFIG_PATH),
        *map(str, arguments),
    ]
    print("$", " ".join(command))
    return subprocess.run(command, check=check, env=planner_subprocess_env())

run_planner_script(
    "check_planner_environment.py",
    "--gpu-test",
    "--output",
    PROJECT_ROOT / "outputs" / "environment" / "planner_environment_preload.json",
)


## 5. 下载固定 revision 的官方 9B BF16 checkpoint

下载量较大。Hugging Face Hub 支持断点续传；如果 Xet 后端卡住，可在执行前设置环境变量 HF_HUB_DISABLE_XET=1。

In [ ]:
DOWNLOAD_MODEL = True

if DOWNLOAD_MODEL:
    run_planner_script(
        "download_planner_checkpoint.py",
        "--checkpoint-dir",
        CHECKPOINT_DIR,
        "--record",
        PROJECT_ROOT / "outputs" / "environment" / "planner_checkpoint_manifest.json",
    )
else:
    print("已跳过下载；后续模型加载与推理不会执行。")


## 6. 最小模型加载检查

默认要求至少 24 GiB 显存，低于该值会在加载前停止。A100 80GB 是本项目优先运行时。此单元只加载模型，不做推理；子进程退出后会释放显存。

In [ ]:
RUN_MODEL_LOAD_CHECK = True

if RUN_MODEL_LOAD_CHECK:
    assert (CHECKPOINT_DIR / "model.safetensors.index.json").is_file(), "checkpoint 尚未完整下载"
    run_planner_script(
        "check_planner_environment.py",
        "--gpu-test",
        "--load-model",
        "--model-path",
        CHECKPOINT_DIR,
        "--output",
        PROJECT_ROOT / "outputs" / "environment" / "planner_environment.json",
    )
else:
    print("已跳过模型加载检查。")


## 7. 查看环境记录

只有下面记录显示版本匹配、CUDA 冒烟通过且 model_load.passed 为 true，才进入 02 Notebook。

In [ ]:
record_path = PROJECT_ROOT / "outputs" / "environment" / "planner_environment.json"
assert record_path.is_file(), record_path
record = json.loads(record_path.read_text(encoding="utf-8"))
print(json.dumps(record, ensure_ascii=False, indent=2))
assert record.get("gpu_smoke_test", {}).get("passed") is True
assert record.get("model_load", {}).get("passed") is True
print("01_planner_setup 验收通过，可以运行 02_planner_inference.ipynb。")
